# Hands-on Python: Questions

[Terminology glossary](../GLOSSARY.md) — definitions for the terms used in these exercises.

[Exercise guide](README.md) · [System design](../system-design/README.md) · [Answers](answers.ipynb)

Eight exercises in four sections: ML data and metrics, retrieval and RAG, evaluation and validation, and production utilities.
Use Python 3.10+. All data is embedded, and exercise code uses only the standard library. No API key or external service is needed.

Run setup once, then the implementation and checks for an exercise. Each exercise is independent of the others.
Replace each TODO stub, then run its checks. Run All intentionally stops at the first unsolved NotImplementedError. To attempt an exercise out of order, run setup and just that exercise. Do not edit the checks to make them pass.

Visible checks are not exhaustive. Add edge cases and explain complexity and production limitations.


In [ ]:
import json
import math
import random
from copy import deepcopy


class RetryableError(Exception):
    """A simulated transient failure for exercise 7."""


def assert_raises(expected, function, *args, **kwargs):
    """Assert the exception type and return it for identity checks."""
    try:
        function(*args, **kwargs)
    except expected as error:
        return error
    raise AssertionError(f'Expected {expected.__name__}')


print('Setup ready: standard library only; no network calls or real sleeps.')


## ML data and metrics


### Exercise 1: Group-aware train/validation split

**Key terms:** [Group split](../GLOSSARY.md#group-split) · [Data leakage](../GLOSSARY.md#data-leakage) · [Random seed and RNG](../GLOSSARY.md#random-seed-and-rng) · [Training set](../GLOSSARY.md#training-set) · [Validation set](../GLOSSARY.md#validation-set)

**Difficulty:** Beginner · **Suggested time:** 20 minutes

Implement `split_by_group(rows, group_key, train_fraction=0.75, seed=0)`.
Return `(train_rows, validation_rows)` without mutating the input or global random state.

- Each row is a dictionary. Its group value must be a nonempty string.
- Keep all rows from one group in the same split. Preserve original row order within each output.
- Sort unique groups, shuffle them with a local `random.Random(seed)`, and assign the first
  `floor(number_of_groups * train_fraction)` groups to training, clamped to leave at least one group in each split.
- Raise `ValueError` for fewer than two groups, missing/invalid group values, or a non-finite fraction outside `(0, 1)`.
  The fraction must be an int or float, excluding booleans.
- This controls the fraction of **groups**, not the fraction of rows. Do not promise balanced classes.

**Discuss:** Why can random row splitting leak customer-specific information? When would you split by time instead?


In [ ]:
def split_by_group(rows, group_key, train_fraction=0.75, seed=0):
    # TODO: implement exercise 1.
    raise NotImplementedError('Exercise 1')


In [ ]:
rows = [{'id': i, 'customer': group} for i, group in enumerate(['a', 'b', 'a', 'c', 'd', 'b'])]
original = deepcopy(rows)
random_state = random.getstate()
train, validation = split_by_group(rows, 'customer', 0.5, seed=7)
assert len({r['customer'] for r in train}) == 2
assert {r['customer'] for r in train}.isdisjoint(r['customer'] for r in validation)
assert sorted(r['id'] for r in train + validation) == list(range(6))
assert [r['id'] for r in train] == sorted(r['id'] for r in train)
assert [r['id'] for r in validation] == sorted(r['id'] for r in validation)
assert (train, validation) == split_by_group(rows, 'customer', 0.5, seed=7)
assert rows == original and random.getstate() == random_state
for fraction in (0.01, 0.99):
    a, b = split_by_group(rows, 'customer', fraction)
    assert a and b
for fraction in (0, 1, float('nan'), True):
    assert_raises(ValueError, split_by_group, rows, 'customer', fraction)
assert_raises(ValueError, split_by_group, [], 'customer')
assert_raises(ValueError, split_by_group, [{'customer': 'a'}], 'customer')
assert_raises(ValueError, split_by_group, [{'missing': 'x'}, {'customer': 'a'}], 'customer')
print('Exercise 1 checks passed')


### Exercise 2: Binary classification metrics

**Key terms:** [Confusion matrix](../GLOSSARY.md#confusion-matrix) · [Accuracy](../GLOSSARY.md#accuracy) · [Precision and recall](../GLOSSARY.md#precision-and-recall) · [F1 score](../GLOSSARY.md#f1-score)

**Difficulty:** Beginner · **Suggested time:** 15 minutes

Implement `binary_metrics(y_true, y_pred)`. Inputs are nonempty equal-length sequences of integer labels 0 or 1
(booleans are not accepted). Raise `ValueError` for invalid labels, empty inputs, or different lengths.
Return a dictionary with `tp`, `tn`, `fp`, `fn`, `accuracy`, `precision`, `recall`, and `f1`.
For a zero denominator, define the affected metric as `0.0`.

**Example:** Truth `[1, 0, 1, 0]` and predictions `[1, 1, 0, 0]` give one of each confusion-matrix count and all four rates equal to 0.5.

**Discuss:** Why can high accuracy be misleading for rare positives? Which errors matter most for your task?


In [ ]:
def binary_metrics(y_true, y_pred):
    # TODO: implement exercise 2.
    raise NotImplementedError('Exercise 2')


In [ ]:
result = binary_metrics([1, 0, 1, 0], [1, 1, 0, 0])
assert result == dict(tp=1, tn=1, fp=1, fn=1, accuracy=0.5, precision=0.5, recall=0.5, f1=0.5)
result = binary_metrics([1, 1, 1, 0], [1, 0, 0, 0])
assert result['precision'] == 1.0 and math.isclose(result['recall'], 1 / 3)
assert result['f1'] == 0.5
result = binary_metrics([0, 0], [0, 0])
assert result['accuracy'] == 1 and result['precision'] == result['recall'] == result['f1'] == 0
assert binary_metrics([1], [1])['f1'] == 1
for actual, predicted in [([], []), ([1], []), ([2], [1]), ([1], [True])]:
    assert_raises(ValueError, binary_metrics, actual, predicted)
print('Exercise 2 checks passed')


## Retrieval and RAG


### Exercise 3: Overlapping document chunks

**Key terms:** [Chunk and overlap](../GLOSSARY.md#chunk-and-overlap) · [Token and tokenizer](../GLOSSARY.md#token-and-tokenizer) · [Big-O notation](../GLOSSARY.md#big-o-notation)

**Difficulty:** Beginner · **Suggested time:** 20 minutes

Implement `chunk_document(doc_id, text, chunk_size, overlap)` using `text.split()` whitespace terms, **not model tokens**.
Return dictionaries with `id` (`doc_id:0`, `doc_id:1`, ...), `text`, `start`, and `end`.
Offsets refer to the whitespace-term list: start inclusive, end exclusive. Join chunk text with single spaces.

- `doc_id` must be a nonempty string.
- `chunk_size` must be a positive integer and `overlap` an integer in `[0, chunk_size)`; reject booleans.
- Raise `ValueError` for invalid configuration even if text is empty. Valid blank text returns `[]`.
- Move forward by `chunk_size - overlap`. Stop when a chunk reaches the end; do not emit a redundant tail.
- A short document produces one chunk.

**Discuss:** What metadata would you retain to cite the original document? Why are whitespace terms an imperfect token budget?


In [ ]:
def chunk_document(doc_id, text, chunk_size, overlap):
    # TODO: implement exercise 3.
    raise NotImplementedError('Exercise 3')


In [ ]:
chunks = chunk_document('d1', 'a b c d e f g', 4, 1)
assert chunks == [dict(id='d1:0', text='a b c d', start=0, end=4),
                  dict(id='d1:1', text='d e f g', start=3, end=7)]
assert [c['text'] for c in chunk_document('d', 'a b c d e', 3, 0)] == ['a b c', 'd e']
assert len(chunk_document('d', 'a b c', 3, 2)) == 1
assert chunk_document('d', ' a\n b ', 4, 1)[0]['text'] == 'a b'
assert chunk_document('d', '   ', 2, 0) == []
for size, overlap in [(0, 0), (3, 3), (3, -1), (True, 0), (3, 0.5)]:
    assert_raises(ValueError, chunk_document, 'd', '', size, overlap)
assert_raises(ValueError, chunk_document, '', 'a', 2, 0)
print('Exercise 3 checks passed')


### Exercise 4: Authorized cosine-similarity retrieval

**Key terms:** [Dot product and cosine similarity](../GLOSSARY.md#dot-product-and-cosine-similarity) · [Embedding](../GLOSSARY.md#embedding) · [Authentication and authorization](../GLOSSARY.md#authentication-and-authorization) · [Tenant](../GLOSSARY.md#tenant)

**Difficulty:** Intermediate · **Suggested time:** 30 minutes

Implement `cosine_similarity(a, b)` and `semantic_search(query, documents, tenant_id, k=3)`.
Vectors are nonempty equal-length lists of finite int/float values, excluding booleans. Cosine returns 0.0 if either vector has zero norm.
Raise `ValueError` for invalid vectors. Samples use modest magnitudes; handling floating-point overflow is outside this exercise.

Documents contain `id`, `tenant_id`, `text`, and `embedding`; IDs are unique within a tenant and other fields are well formed.
Filter by the trusted `tenant_id` **before reading/scoring document embeddings**. Validate the query even when there are no matches.
Return up to `k` dictionaries with `id`, `text`, and `score`, sorted by decreasing score, breaking ties by ascending ID.
`k` must be a nonnegative integer excluding booleans; invalid k raises `ValueError`. `k=0` returns `[]`.
Do not mutate inputs. All documents eligible for scoring must have valid matching-dimension vectors.

These tiny fixed vectors illustrate ranking; they are not embeddings generated by a model.
**Discuss:** Why is a tenant filter in an interview exercise not a complete production authorization system?


In [ ]:
def cosine_similarity(a, b):
    # TODO: implement exercise 4.
    raise NotImplementedError('Exercise 4')


def semantic_search(query, documents, tenant_id, k=3):
    # TODO: implement exercise 4.
    raise NotImplementedError('Exercise 4')


In [ ]:
assert math.isclose(cosine_similarity([1, 1], [1, 0]), 1 / math.sqrt(2))
assert cosine_similarity([1, 0], [-1, 0]) == -1
assert cosine_similarity([0, 0], [1, 0]) == 0
for a, b in [([], []), ([1], [1, 2]), ([float('nan')], [1]), ([True], [1])]:
    assert_raises(ValueError, cosine_similarity, a, b)
documents = [
    dict(id='b', tenant_id='t1', text='second tie', embedding=[1, 0]),
    dict(id='a', tenant_id='t1', text='first tie', embedding=[2, 0]),
    dict(id='c', tenant_id='t1', text='orthogonal', embedding=[0, 1]),
    dict(id='private', tenant_id='t2', text='not eligible', embedding=[99]),
]
before = deepcopy(documents)
matches = semantic_search([1, 0], documents, 't1', 2)
assert [r['id'] for r in matches] == ['a', 'b']
assert all(r['score'] == 1.0 for r in matches)
assert documents == before
assert semantic_search([1, 0], documents, 'missing') == []
assert semantic_search([1, 0], documents, 't1', 0) == []
assert len(semantic_search([1, 0], documents, 't1', 20)) == 3
assert_raises(ValueError, semantic_search, [], [], 't1')
assert_raises(ValueError, semantic_search, [1, 0], documents, 't1', -1)
print('Exercise 4 checks passed')


## Evaluation and validation


### Exercise 5: Retrieval metrics at a cutoff

**Key terms:** [Precision@k and recall@k](../GLOSSARY.md#precisionk-and-recallk) · [Precision@k and recall@k](../GLOSSARY.md#precisionk-and-recallk) · [Mean reciprocal rank (MRR)](../GLOSSARY.md#mean-reciprocal-rank-mrr)

**Difficulty:** Intermediate · **Suggested time:** 20 minutes

Implement `retrieval_metrics(retrieved_ids, relevant_ids, k)` for one query.
Inputs contain hashable string IDs; retrieved IDs must be unique across the entire input list or raise `ValueError`.
Treat relevant IDs as a set. `k` must be a positive integer excluding booleans.

Return `precision_at_k`, `recall_at_k`, and `reciprocal_rank` as floats.
Precision uses denominator k **even when fewer than k results are returned**. Recall uses the number of distinct relevant IDs,
and is defined as 0.0 when there are none. Reciprocal rank is the inverse of the first relevant rank within the top k, or 0.0.
Empty retrieval is valid. This is single-query reciprocal rank; MRR requires averaging over queries.

**Discuss:** Why should the cutoff and empty-reference convention accompany every reported score?


In [ ]:
def retrieval_metrics(retrieved_ids, relevant_ids, k):
    # TODO: implement exercise 5.
    raise NotImplementedError('Exercise 5')


In [ ]:
assert retrieval_metrics(['x', 'a', 'y', 'z', 'b'], {'a', 'b', 'c', 'd'}, 5) == {
    'precision_at_k': 0.4, 'recall_at_k': 0.5, 'reciprocal_rank': 0.5}
assert retrieval_metrics(['a'], ['a', 'a'], 4) == {
    'precision_at_k': 0.25, 'recall_at_k': 1.0, 'reciprocal_rank': 1.0}
assert retrieval_metrics(['x', 'a'], {'a'}, 1)['reciprocal_rank'] == 0
assert retrieval_metrics([], {'a'}, 3) == dict(precision_at_k=0.0, recall_at_k=0.0, reciprocal_rank=0.0)
assert retrieval_metrics(['a'], [], 1)['recall_at_k'] == 0
assert_raises(ValueError, retrieval_metrics, ['a', 'a'], {'a'}, 1)
assert_raises(ValueError, retrieval_metrics, ['a'], {'a'}, 0)
assert_raises(ValueError, retrieval_metrics, ['a'], {'a'}, True)
print('Exercise 5 checks passed')


### Exercise 6: Validate a structured model response

**Key terms:** [Structured output and schema](../GLOSSARY.md#structured-output-and-schema) · [Structured output and schema](../GLOSSARY.md#structured-output-and-schema) · [Deterministic check](../GLOSSARY.md#deterministic-check)

**Difficulty:** Intermediate · **Suggested time:** 25 minutes

Implement `parse_ticket(response_text)` using the standard-library JSON parser. Return a new dictionary with
exactly the keys `category`, `priority`, and `summary`, or raise `ValueError`.

- Input must decode to one JSON object, not a list or a string. Do not extract JSON from prose or code fences.
- Reject missing/extra keys and duplicate JSON keys.
- Category must be one of `billing`, `technical`, or `other`.
- Priority must be an integer 1, 2, or 3. A JSON boolean is invalid even though Python's bool subclasses int.
- Summary must be a nonempty string after trimming whitespace. Return the trimmed summary.
- Malformed JSON, trailing prose, and invalid field values must fail explicitly.

**Discuss:** What can this validator establish, and what would need evidence or business rules to verify?


In [ ]:
def parse_ticket(response_text):
    # TODO: implement exercise 6.
    raise NotImplementedError('Exercise 6')


In [ ]:
good = dict(category='billing', priority=2, summary='  Duplicate charge  ')
assert parse_ticket(json.dumps(good)) == dict(category='billing', priority=2, summary='Duplicate charge')
invalid = [
    'not json', '[]', '{}', json.dumps(good) + ' trailing text',
    json.dumps({**good, 'priority': True}), json.dumps({**good, 'priority': 2.0}),
    json.dumps({**good, 'priority': 4}), json.dumps({**good, 'summary': '  '}),
    json.dumps({**good, 'summary': None}), json.dumps({**good, 'category': 'unknown'}),
    json.dumps({**good, 'category': []}), json.dumps({**good, 'extra': 'field'}),
    '{"category":"billing","category":"other","priority":1,"summary":"x"}',
]
for raw in invalid:
    assert_raises(ValueError, parse_ticket, raw)
print('Exercise 6 checks passed')


## Production utilities


### Exercise 7: Bounded retries for a read operation

**Key terms:** [Retry and backoff](../GLOSSARY.md#retry-and-backoff) · [Idempotency](../GLOSSARY.md#idempotency) · [Deadline and cancellation](../GLOSSARY.md#deadline-and-cancellation)

**Difficulty:** Intermediate · **Suggested time:** 20 minutes

Implement `retry_read(operation, max_attempts=3)`. The zero-argument callable represents an idempotent read.
`RetryableError` is provided in setup. Return `(result, attempts_used)` on success, including when result is `None`.

- `max_attempts` is the total number of calls allowed, including the first; require a positive integer excluding booleans.
- Retry only `RetryableError`. Propagate every other exception immediately.
- When the budget is exhausted, propagate the last retryable exception itself.
- Validate the budget before calling the operation. Do not sleep or call an external service in this exercise.

**Discuss:** Where would deadlines, jitter, cancellation, and idempotency matter in a real retry policy?


In [ ]:
def retry_read(operation, max_attempts=3):
    # TODO: implement exercise 7.
    raise NotImplementedError('Exercise 7')


In [ ]:
calls = []
def flaky_read():
    calls.append('called')
    if len(calls) < 3:
        raise RetryableError('temporary')
    return 'ready'
assert retry_read(flaky_read, 3) == ('ready', 3) and len(calls) == 3
assert retry_read(lambda: None, 1) == (None, 1)
calls = []
marker = RetryableError('still unavailable')
def unavailable():
    calls.append('called')
    raise marker
assert assert_raises(RetryableError, retry_read, unavailable, 2) is marker
assert len(calls) == 2
calls = []
def invalid_request():
    calls.append('called')
    raise ValueError('invalid request')
assert_raises(ValueError, retry_read, invalid_request, 5)
assert len(calls) == 1
for limit in (0, -1, True):
    calls.clear()
    assert_raises(ValueError, retry_read, unavailable, limit)
    assert calls == []
print('Exercise 7 checks passed')


### Exercise 8: Authorization-scoped TTL response cache

**Key terms:** [Time to live (TTL)](../GLOSSARY.md#time-to-live-ttl) · [Response, semantic, and prefix caches](../GLOSSARY.md#response-semantic-and-prefix-caches) · [Authentication and authorization](../GLOSSARY.md#authentication-and-authorization) · [Tenant](../GLOSSARY.md#tenant)

**Difficulty:** Intermediate · **Suggested time:** 30 minutes

Implement `ScopedTTLCache(ttl_seconds)` with the `put` and `get` signatures below. Values are strings;
`None` means a cache miss. A valid empty string must remain a cache hit.

- The key consists of the exact strings `(scope, model_version, data_version, prompt_version, query)`.
  `scope` is a trusted authorization partition, such as tenant plus permission revision; it is not raw user input.
- TTL must be a finite positive int/float, excluding booleans. `now` must be finite, nonnegative int/float, excluding booleans.
  Invalid TTL/time or a non-string value raises `ValueError`. Assume key components are valid strings.
- `put` stores/replaces an entry with expiry `now + ttl_seconds`.
- `get` returns the value only when `now < expiry`. At exact expiry, remove the entry and return `None`.
- Caller timestamps use the same monotonically advancing clock. Do not read real time or sleep.

This is an exact-response cache, not prefix or semantic caching.
**Discuss:** How would eviction, permission revocation, concurrent access, and multiple worker processes change the design?


In [ ]:
class ScopedTTLCache:
    def __init__(self, ttl_seconds):
        # TODO: implement exercise 8.
        raise NotImplementedError('Exercise 8')

    def put(self, scope, model_version, data_version, prompt_version, query, value, now):
        raise NotImplementedError('Exercise 8')

    def get(self, scope, model_version, data_version, prompt_version, query, now):
        raise NotImplementedError('Exercise 8')


In [ ]:
key = dict(scope='tenant-a:permission-v1', model_version='m1', data_version='d1',
           prompt_version='p1', query='refund policy')
cache = ScopedTTLCache(5)
assert cache.get(**key, now=0) is None
cache.put(**key, value='30 days', now=10)
assert cache.get(**key, now=14.999) == '30 days'
for field in key:
    changed = {**key, field: key[field] + '-changed'}
    assert cache.get(**changed, now=14.999) is None
assert cache.get(**key, now=15) is None
cache.put(**key, value='', now=20)
assert cache.get(**key, now=21) == ''
cache.put(**key, value='updated', now=22)
assert cache.get(**key, now=26) == 'updated'
assert cache.get(**key, now=27) is None
for ttl in (0, -1, float('inf'), float('nan'), True):
    assert_raises(ValueError, ScopedTTLCache, ttl)
for now in (-1, float('nan'), float('inf'), True):
    assert_raises(ValueError, cache.get, **key, now=now)
    assert_raises(ValueError, cache.put, **key, value='x', now=now)
assert_raises(ValueError, cache.put, **key, value=None, now=30)
print('Exercise 8 checks passed')


## Review and extensions

- Add a failing edge case before improving an implementation.
- Explain how data, identity, or configuration changes affect correctness.
- Connect the functions to the [system design scenarios](../system-design/README.md).
- For background, review [AI foundations](../questions/ai-foundations.md), [RAG](../questions/rag.md), [evaluation](../questions/evaluation.md), and [production engineering](../questions/production-engineering.md).

Useful standard-library references: [JSON parsing](https://docs.python.org/3/library/json.html) and [local random generators](https://docs.python.org/3/library/random.html).
